# 03 – Model Selection and Development (Stage 5)
Baseline + four models. One stratified 80/20 split; the 20% test set is NOT touched here.
Comparison = 3-fold CV on a 40,000-row sample of the TRAINING set (compute budget).

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import time
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from common import *

df = add_features(pd.read_csv(CLEAN_PATH, low_memory=False))
train, test = train_test_split(df, test_size=0.20, stratify=df[TARGET], random_state=RANDOM_STATE)
train.to_csv(ROOT / "data" / "processed" / "train.csv", index=False)
test.to_csv(ROOT / "data" / "processed" / "test.csv", index=False)
print("train", train.shape, "test", test.shape)
sub, _ = train_test_split(train, train_size=40000, stratify=train[TARGET], random_state=RANDOM_STATE)


def make_pipe(model, num, cat, scale_onehot):
    if scale_onehot:     # linear model: scale numbers, one-hot categories
        n = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())])
        c = Pipeline([("i", SimpleImputer(strategy="most_frequent")), ("o", OneHotEncoder(handle_unknown="ignore"))])
    else:                # tree models: no scaling needed
        n = SimpleImputer(strategy="median")
        c = Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                      ("o", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    return Pipeline([("pre", ColumnTransformer([("num", n, num), ("cat", c, cat)])), ("model", model)])


MODELS = {
    "Baseline (most frequent)": (lambda: DummyClassifier(strategy="most_frequent"), True),
    "Logistic Regression": (lambda: LogisticRegression(max_iter=300, class_weight="balanced"), True),
    "Decision Tree": (lambda: DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE), False),
    "Random Forest": (lambda: RandomForestClassifier(n_estimators=100, max_depth=14, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE), False),
    "Gradient Boosting (HGB)": (lambda: HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, class_weight="balanced", random_state=RANDOM_STATE), False),
}
cv = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)
rows = []
for use_eng in (False, True):
    feats, num, cat = split_columns(sub, use_eng)
    for name, (mk, onehot) in MODELS.items():
        t0 = time.time()
        pred = cross_val_predict(make_pipe(mk(), num, cat, onehot), sub[feats], sub[TARGET], cv=cv)
        y = sub[TARGET]
        rows.append({"model": name, "features": "raw + engineered" if use_eng else "raw only", "n_features": len(feats),
                     "accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
                     "weighted_f1": f1_score(y, pred, average="weighted"),
                     "at_risk_recall": recall_score(y.isin(AT_RISK), pd.Series(pred, index=y.index).isin(AT_RISK)),
                     "seconds": round(time.time() - t0, 1)})
        print(rows[-1], flush=True)
res = pd.DataFrame(rows)
res.to_csv(RES_DIR / "model_comparison_cv.csv", index=False)
print(res.round(4).to_string())